# CS781 Capstone Phase 7: Prior Case Retrieval (PCR)
## Submission 01: Model 4 Full Fine-Tuning Hierarchical Bi-Encoder (~153.9M)

* **Task**: Prior Case Retrieval (PCR) — Retrieve Top 7 legally & semantically relevant prior cases for query legal cases.
* **Primary Target Metric**: **Micro-F1@7** (Secondary: Micro-Precision@7, Micro-Recall@7).
* **Base Foundation Model**: `sarweshbarnwal/CS781-Legal-Models` -> `model4-capacity-boundary-154m` (Pretrained 100% from scratch in Phase 1; LLaMA-style, 14 layers, $d=896$, Tied Embeddings).
* **Architecture**: Two-Tower Legal Bi-Encoder with a 2-layer Hierarchical Cross-Chunk Transformer Attention Pooler ($L_{\text{chunk}}=512$, stride $=256$, max 8 chunks).
* **Training Objective**: InfoNCE / Symmetric Multiple Negatives Ranking Loss ($\tau = 0.05$) using in-batch negatives.
* **Optimization**: AdamW (Peak LR $2.0 \times 10^{-5}$, Cosine Annealing, 500 warmup steps).
* **Hardware Utilization**: Accelerated on Dual NVIDIA T4 GPUs (`cuda:0` and `cuda:1`) with PyTorch DataParallel and SDPA.
* **Checkpoint & Cloud Resilience**: Continuous non-blocking push to Hugging Face Hub every 200 steps with automatic crash recovery.


In [ ]:
# 1. Environment Dependencies & Hugging Face Authentication
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
import gc
import json
import time
import math
import glob
import random
import zipfile
import shutil
import threading
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Defensive PyTorch & Transformers compatibility guard for Python 3.13 / Kaggle
import sys
import types
import importlib
import torch

try:
    _u = importlib.import_module("torch._utils")
except Exception:
    _u = types.ModuleType("torch._utils")

if not hasattr(_u, "_get_device_index"):
    def _get_device_index(device, optional=False, allow_cpu=False):
        if isinstance(device, str): device = torch.device(device)
        if isinstance(device, torch.device): return -1 if device.type == "cpu" else (device.index or 0)
        if isinstance(device, int): return device
        return 0
    _u._get_device_index = _get_device_index

if not hasattr(_u, "try_import"):
    def try_import(name):
        try:
            return importlib.import_module(name)
        except Exception:
            return None
    _u.try_import = try_import

def _fallback_getattr(name):
    if name == "try_import":
        return _u.try_import
    if name == "_get_device_index":
        return _u._get_device_index
    return lambda *args, **kwargs: None

_u.__getattr__ = _fallback_getattr
setattr(torch, "_utils", _u)
sys.modules["torch._utils"] = _u

try:
    import transformers.utils.import_utils as hf_import_utils
    hf_import_utils.is_torchvision_available = lambda: False
except Exception:
    pass

try:
    import transformers.distributed.checkpoint as hf_dist_cp
    hf_dist_cp._check_distributed_checkpointing_available = lambda: False
except Exception:
    pass

from huggingface_hub import HfApi, login, hf_hub_download, snapshot_download

# Authenticate via Kaggle Secrets or Environment
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("✓ Hugging Face authentication successful.")
else:
    print("⚠ WARNING: HF_TOKEN not found. Ensure it is added in Kaggle Secrets.")


In [ ]:
# 2. Execution Profile Controller (CORE vs. SMOKE)
# Change PROFILE to "SMOKE" for a rapid 2-minute pipeline verification, or "CORE" for full production execution.
PROFILE = "CORE"  # Options: "CORE" or "SMOKE"

PROFILES = {
    "SMOKE": {
        "max_train_queries": 24,
        "max_candidates": 80,
        "max_dev_queries": 16,
        "max_test_queries": 20,
        "train_epochs": 1,
        "batch_size": 2,
        "eval_steps": 20,
        "save_steps": 20,
        "warmup_steps": 10,
        "max_chunks": 4,
        "chunk_size": 256,
        "chunk_stride": 128,
        "description": "Rapid Sanity Verification (Smoke Mode)"
    },
    "CORE": {
        "max_train_queries": None,    # Full dataset
        "max_candidates": None,       # Full candidate pool
        "max_dev_queries": None,      # Full dev validation
        "max_test_queries": None,     # Full test set
        "train_epochs": 4,
        "batch_size": 8,              # Micro-batch per step (scaled via gradient accum)
        "eval_steps": 200,
        "save_steps": 200,            # Continuous cloud sync every 200 steps
        "warmup_steps": 500,
        "max_chunks": 8,              # Up to 2304 tokens of legal context
        "chunk_size": 512,
        "chunk_stride": 256,
        "description": "Full Production Multi-GPU Training (Core Mode)"
    }
}

CFG = PROFILES[PROFILE]
print(f"=== Active Profile: {PROFILE} ({CFG['description']}) ===")

# Global Constants & Paths
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MASTER_REPO = "sarweshbarnwal/CS781-Legal-Models"
BASE_MODEL_SUBFOLDER = "model4-capacity-boundary-154m"
EXPERIMENT_NAME = "sub01_model4_full_ft_hierarchical"
HUB_SUBFOLDER = f"Phase_7/{EXPERIMENT_NAME}"

OUTPUT_DIR = "/kaggle/working/phase7_sub01" if os.path.exists("/kaggle") else "./phase7_sub01"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "loss_and_metrics_plot.png")
METRICS_JSON_PATH = os.path.join(OUTPUT_DIR, "val_metrics.json")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Local artifacts directory: {OUTPUT_DIR}")
print(f"HF Hub destination: {HF_MASTER_REPO}/{HUB_SUBFOLDER}")


In [ ]:
# 3. Hardware Acceleration & Dual-T4 Multi-GPU Setup
print("=== Hardware Acceleration Verification ===")
gpu_count = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | GPU Count: {gpu_count}")

if torch.cuda.is_available():
    primary_device = torch.device("cuda:0")
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU [{i}]: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB")
    
    # Enable PyTorch SDPA memory-efficient kernels optimized for NVIDIA T4
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cuda.enable_flash_sdp(False) # T4 Tensor Cores use memory-efficient SDPA
    torch.backends.cudnn.benchmark = True
    print("✓ PyTorch 2.0 SDPA memory-efficient attention activated.")
else:
    primary_device = torch.device("cpu")
    print("⚠ No GPU detected. Running on CPU fallback.")

device = primary_device
USE_MULTI_GPU = (gpu_count > 1)
print(f"Primary Compute Device: {device} | Dual-GPU Parallelization: {USE_MULTI_GPU}")


In [ ]:
# 4. Ingest Phase 7 Dataset from Hugging Face
print("Downloading Phase 7 PCR dataset splits...")

def load_pcr_json(filename: str):
    fpath = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=f"Phase_7_PCR/{filename}",
        repo_type="dataset",
        token=HF_TOKEN
    )
    with open(fpath, "r", encoding="utf-8") as f:
        return json.load(f)

# Load raw JSON structures
raw_train_queries = load_pcr_json("Train/train.json")
raw_train_cands = load_pcr_json("Train/train_candidates.json")
raw_dev_queries = load_pcr_json("Dev/dev.json")
raw_dev_cands = load_pcr_json("Dev/dev_candidates.json")
raw_test_queries = load_pcr_json("Test/test.json")
raw_test_cands = load_pcr_json("Test/test_candidates.json")

def slice_queries_and_candidates(raw_queries: dict, raw_cands: dict, max_queries: Optional[int], max_candidates: Optional[int]):
    """
    Slices queries and candidates while guaranteeing that all gold relevant candidates
    for the selected queries are present in the candidate pool.
    """
    if max_queries is not None:
        selected_queries = {k: raw_queries[k] for k in list(raw_queries.keys())[:max_queries]}
    else:
        selected_queries = raw_queries

    if max_candidates is None:
        return selected_queries, raw_cands

    # Collect all gold candidate IDs needed by the selected queries
    gold_cids = set()
    for qdata in selected_queries.values():
        if isinstance(qdata, dict):
            rels = qdata.get("relevant_candidates", qdata.get("gold_candidates", []))
            if isinstance(rels, list):
                gold_cids.update(rels)
            elif isinstance(rels, str):
                gold_cids.add(rels)

    # Prioritize gold candidates present in raw_cands, then fill up to max_candidates with distractors
    selected_cids = [cid for cid in raw_cands.keys() if cid in gold_cids]
    for cid in raw_cands.keys():
        if len(selected_cids) >= max_candidates:
            break
        if cid not in gold_cids:
            selected_cids.append(cid)

    # Safety: ensure at least 7 candidates for evaluation
    if len(selected_cids) < 7:
        for cid in raw_cands.keys():
            if cid not in selected_cids:
                selected_cids.append(cid)
            if len(selected_cids) >= min(7, len(raw_cands)):
                break

    selected_cands = {cid: raw_cands[cid] for cid in selected_cids}
    return selected_queries, selected_cands

train_queries, train_cands = slice_queries_and_candidates(raw_train_queries, raw_train_cands, CFG["max_train_queries"], CFG["max_candidates"])
dev_queries, dev_cands = slice_queries_and_candidates(raw_dev_queries, raw_dev_cands, CFG["max_dev_queries"], CFG["max_candidates"])
test_queries, test_cands = slice_queries_and_candidates(raw_test_queries, raw_test_cands, CFG["max_test_queries"], CFG["max_candidates"])

print(f"\nActive Profile ({PROFILE}) Ingested Counts:")
print(f"  Train Queries: {len(train_queries):,} | Train Candidates: {len(train_cands):,}")
print(f"  Dev Queries:   {len(dev_queries):,} | Dev Candidates:   {len(dev_cands):,}")
print(f"  Test Queries:  {len(test_queries):,} | Test Candidates:  {len(test_cands):,}")


In [ ]:
# Defensive PyTorch & Transformers compatibility guard for Python 3.13 / Kaggle
import sys
import types
import importlib
import torch

try:
    _u = importlib.import_module("torch._utils")
except Exception:
    _u = types.ModuleType("torch._utils")

if not hasattr(_u, "_get_device_index"):
    def _get_device_index(device, optional=False, allow_cpu=False):
        if isinstance(device, str): device = torch.device(device)
        if isinstance(device, torch.device): return -1 if device.type == "cpu" else (device.index or 0)
        if isinstance(device, int): return device
        return 0
    _u._get_device_index = _get_device_index

if not hasattr(_u, "try_import"):
    def try_import(name):
        try:
            return importlib.import_module(name)
        except Exception:
            return None
    _u.try_import = try_import

def _fallback_getattr(name):
    if name == "try_import":
        return _u.try_import
    if name == "_get_device_index":
        return _u._get_device_index
    return lambda *args, **kwargs: None

_u.__getattr__ = _fallback_getattr
setattr(torch, "_utils", _u)
sys.modules["torch._utils"] = _u

try:
    import transformers.utils.import_utils as hf_import_utils
    hf_import_utils.is_torchvision_available = lambda: False
except Exception:
    pass

try:
    import transformers.distributed.checkpoint as hf_dist_cp
    hf_dist_cp._check_distributed_checkpointing_available = lambda: False
except Exception:
    pass

# 5. Load Custom Domain-Trained 32K Legal Tokenizer
from transformers import PreTrainedTokenizerFast

print("Loading custom 32,000 Byte-Level BPE legal tokenizer...")
# Download tokenizer artifacts from Master Repo
tok_files = ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"]
local_tok_dir = os.path.join(OUTPUT_DIR, "tokenizer")
os.makedirs(local_tok_dir, exist_ok=True)

for tf in tok_files:
    try:
        hf_hub_download(
            repo_id=HF_MASTER_REPO,
            filename=f"{BASE_MODEL_SUBFOLDER}/tokenizer/{tf}",
            local_dir=local_tok_dir,
            token=HF_TOKEN
        )
    except Exception:
        # Fallback to direct subfolder root
        try:
            hf_hub_download(
                repo_id=HF_MASTER_REPO,
                filename=f"{BASE_MODEL_SUBFOLDER}/{tf}",
                local_dir=local_tok_dir,
                token=HF_TOKEN
            )
        except Exception as e:
            pass

# Instantiate tokenizer
tok_json_path = os.path.join(local_tok_dir, BASE_MODEL_SUBFOLDER, "tokenizer", "tokenizer.json")
if not os.path.exists(tok_json_path):
    tok_json_path = os.path.join(local_tok_dir, BASE_MODEL_SUBFOLDER, "tokenizer.json")
if not os.path.exists(tok_json_path):
    tok_json_path = os.path.join(local_tok_dir, "tokenizer.json")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=tok_json_path,
    bos_token="<bos>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>"
)
tokenizer.padding_side = "right"
print(f"✓ Tokenizer loaded successfully. Vocab Size: {len(tokenizer):,}")
print(f"  Special tokens: PAD={tokenizer.pad_token_id}, BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}")


In [ ]:
# 6. Two-Tower Legal Bi-Encoder with Hierarchical Cross-Chunk Attention
# Defensive PyTorch & Transformers compatibility guard for Python 3.13 / Kaggle
import sys
import types
import importlib
import torch

try:
    _u = importlib.import_module("torch._utils")
except Exception:
    _u = types.ModuleType("torch._utils")

if not hasattr(_u, "_get_device_index"):
    def _get_device_index(device, optional=False, allow_cpu=False):
        if isinstance(device, str): device = torch.device(device)
        if isinstance(device, torch.device): return -1 if device.type == "cpu" else (device.index or 0)
        if isinstance(device, int): return device
        return 0
    _u._get_device_index = _get_device_index

if not hasattr(_u, "try_import"):
    def try_import(name):
        try:
            return importlib.import_module(name)
        except Exception:
            return None
    _u.try_import = try_import

def _fallback_getattr(name):
    if name == "try_import":
        return _u.try_import
    if name == "_get_device_index":
        return _u._get_device_index
    return lambda *args, **kwargs: None

_u.__getattr__ = _fallback_getattr
setattr(torch, "_utils", _u)
sys.modules["torch._utils"] = _u

try:
    import transformers.utils.import_utils as hf_import_utils
    hf_import_utils.is_torchvision_available = lambda: False
except Exception:
    pass

try:
    import transformers.distributed.checkpoint as hf_dist_cp
    hf_dist_cp._check_distributed_checkpointing_available = lambda: False
except Exception:
    pass

from transformers import LlamaConfig, LlamaModel

class HierarchicalChunkPooler(nn.Module):
    """
    Cross-Chunk Transformer Attention Pooler:
    Synthesizes up to K chunk vectors (from sliding window passages) into a single
    dense document representation using learned chunk position embeddings.
    """
    def __init__(self, hidden_size: int = 896, max_chunks: int = 8, nhead: int = 4):
        super().__init__()
        self.chunk_pos_embed = nn.Embedding(max_chunks, hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=nhead,
            dim_feedforward=hidden_size * 2,
            dropout=0.1,
            activation="gelu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, chunk_vectors: torch.Tensor, chunk_mask: torch.Tensor) -> torch.Tensor:
        """
        chunk_vectors: [Batch, K_chunks, hidden_size]
        chunk_mask:    [Batch, K_chunks] (1 for valid chunk, 0 for padded chunk)
        """
        B, K, D = chunk_vectors.shape
        positions = torch.arange(K, device=chunk_vectors.device).unsqueeze(0).expand(B, K)
        x = chunk_vectors + self.chunk_pos_embed(positions)

        # PyTorch TransformerEncoder treats True as masked positions
        key_pad_mask = (chunk_mask == 0)
        out = self.transformer(x, src_key_padding_mask=key_pad_mask)

        # Attention-masked average pooling across valid chunks
        mask_expanded = chunk_mask.unsqueeze(-1).float()
        pooled = (out * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1e-9)
        return self.norm(pooled)


class LegalBiEncoder(nn.Module):
    """
    Two-Tower Legal Bi-Encoder:
    Maps long legal cases (queries & candidates) into L2-normalized dense embeddings.
    """
    def __init__(self, backbone: LlamaModel, hidden_size: int = 896, max_chunks: int = 8):
        super().__init__()
        self.backbone = backbone
        self.hidden_size = hidden_size
        self.chunk_pooler = HierarchicalChunkPooler(hidden_size=hidden_size, max_chunks=max_chunks)

    def encode_chunks(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, chunk_mask: torch.Tensor) -> torch.Tensor:
        """
        input_ids:      [Batch, K_chunks, L_chunk]
        attention_mask: [Batch, K_chunks, L_chunk]
        chunk_mask:     [Batch, K_chunks]
        """
        B, K, L = input_ids.shape
        flat_ids = input_ids.view(B * K, L)
        flat_mask = attention_mask.view(B * K, L)

        # Micro-sliced backbone pass (max 8 chunks per forward) to eliminate peak VRAM spikes
        chunk_reprs_list = []
        max_sub_batch = 4
        for s_idx in range(0, flat_ids.size(0), max_sub_batch):
            e_idx = min(s_idx + max_sub_batch, flat_ids.size(0))
            sub_ids = flat_ids[s_idx:e_idx]
            sub_mask = flat_mask[s_idx:e_idx]

            sub_out = self.backbone(input_ids=sub_ids, attention_mask=sub_mask)
            sub_hidden = sub_out.last_hidden_state

            sub_mask_exp = sub_mask.unsqueeze(-1).float()
            sub_repr = (sub_hidden * sub_mask_exp).sum(dim=1) / sub_mask_exp.sum(dim=1).clamp(min=1e-9)
            chunk_reprs_list.append(sub_repr)

        chunk_reprs = torch.cat(chunk_reprs_list, dim=0).view(B, K, -1)  # [B, K, D]

        # Aggregate across chunks
        doc_vector = self.chunk_pooler(chunk_reprs, chunk_mask)
        # Project onto unit hypersphere for stable cosine similarity
        return F.normalize(doc_vector, p=2, dim=-1)

    def forward(self, q_ids, q_attn, q_cmask, d_ids=None, d_attn=None, d_cmask=None):
        if d_ids is not None:
            # Training mode: encode both query and candidate towers
            u = self.encode_chunks(q_ids, q_attn, q_cmask)
            v = self.encode_chunks(d_ids, d_attn, d_cmask)
            return u, v
        else:
            # Inference mode: encode single tower
            return self.encode_chunks(q_ids, q_attn, q_cmask)


print("Loading Phase 1 Model 4 Pretrained Checkpoint...")
# Download model weights from Master Hub
m4_dir = os.path.join(OUTPUT_DIR, "model4_base")
os.makedirs(m4_dir, exist_ok=True)

snapshot_download(
    repo_id=HF_MASTER_REPO,
    allow_patterns=[f"{BASE_MODEL_SUBFOLDER}/*"],
    local_dir=m4_dir,
    token=HF_TOKEN
)

base_subpath = os.path.join(m4_dir, BASE_MODEL_SUBFOLDER)
config = LlamaConfig.from_pretrained(base_subpath)
base_backbone = LlamaModel.from_pretrained(base_subpath, config=config)
base_backbone.gradient_checkpointing_enable()
print("✓ Gradient checkpointing enabled on backbone to conserve VRAM.")

model = LegalBiEncoder(
    backbone=base_backbone,
    hidden_size=config.hidden_size,
    max_chunks=CFG["max_chunks"]
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ LegalBiEncoder instantiated successfully.")
print(f"  Backbone Hidden Dim: {config.hidden_size} | Layers: {config.num_hidden_layers}")
print(f"  Total Parameters: {total_params / 1e6:.2f}M | Trainable: {trainable_params / 1e6:.2f}M")


In [ ]:
# 7. Document Chunking Engine & Multi-Instance Ranking Dataset
def process_text_into_chunks(
    sentences: List[str],
    tokenizer: PreTrainedTokenizerFast,
    chunk_size: int,
    stride: int,
    max_chunks: int
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Converts list of sentence strings into uniform sliding-window chunk tensors.
    """
    full_text = " ".join(sentences)
    tokens = tokenizer.encode(full_text, add_special_tokens=False)
    
    chunk_ids = []
    chunk_masks = []
    
    for start in range(0, min(len(tokens), max_chunks * stride), stride):
        chunk = tokens[start : start + chunk_size]
        pad_len = chunk_size - len(chunk)
        chunk_ids.append(chunk + [tokenizer.pad_token_id] * pad_len)
        chunk_masks.append([1] * len(chunk) + [0] * pad_len)
        if len(chunk_ids) >= max_chunks:
            break
            
    # Handle documents shorter than max_chunks
    actual_k = len(chunk_ids)
    if actual_k == 0:
        chunk_ids.append([tokenizer.pad_token_id] * chunk_size)
        chunk_masks.append([0] * chunk_size)
        actual_k = 1

    while len(chunk_ids) < max_chunks:
        chunk_ids.append([tokenizer.pad_token_id] * chunk_size)
        chunk_masks.append([0] * chunk_size)

    cmask = [1] * actual_k + [0] * (max_chunks - actual_k)
    return (
        torch.tensor(chunk_ids, dtype=torch.long),
        torch.tensor(chunk_masks, dtype=torch.long),
        torch.tensor(cmask, dtype=torch.long)
    )


class PCRTrainDataset(Dataset):
    """
    Pairs each query case with one of its gold relevant candidate cases per epoch.
    """
    def __init__(self, queries: dict, candidates: dict, tokenizer, chunk_size, stride, max_chunks):
        self.samples = []
        self.candidates = candidates
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.stride = stride
        self.max_chunks = max_chunks

        cand_keys = list(candidates.keys())
        for qid, qdata in queries.items():
            if isinstance(qdata, dict):
                rel_cands = qdata.get("relevant_candidates", qdata.get("gold_candidates", []))
                q_text = qdata.get("text", "")
            else:
                rel_cands = []
                q_text = qdata
            if isinstance(rel_cands, str):
                rel_cands = [rel_cands]
            valid_rels = [cid for cid in rel_cands if cid in candidates]
            if valid_rels:
                self.samples.append((qid, q_text, valid_rels))
            elif cand_keys:
                self.samples.append((qid, q_text, [cand_keys[0]]))

        print(f"PCRTrainDataset built with {len(self.samples):,} valid training pairs.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        qid, q_text, valid_rels = self.samples[idx]
        # Randomly sample one positive candidate for this step
        pos_cid = random.choice(valid_rels)
        cand_entry = self.candidates[pos_cid]
        cand_text = cand_entry.get("text", "") if isinstance(cand_entry, dict) else cand_entry

        q_ids, q_attn, q_cmask = process_text_into_chunks(
            q_text, self.tokenizer, self.chunk_size, self.stride, self.max_chunks
        )
        d_ids, d_attn, d_cmask = process_text_into_chunks(
            cand_text, self.tokenizer, self.chunk_size, self.stride, self.max_chunks
        )

        return q_ids, q_attn, q_cmask, d_ids, d_attn, d_cmask

def pcr_collate_fn(batch):
    q_ids = torch.stack([item[0] for item in batch])
    q_attn = torch.stack([item[1] for item in batch])
    q_cmask = torch.stack([item[2] for item in batch])
    d_ids = torch.stack([item[3] for item in batch])
    d_attn = torch.stack([item[4] for item in batch])
    d_cmask = torch.stack([item[5] for item in batch])
    return q_ids, q_attn, q_cmask, d_ids, d_attn, d_cmask

train_dataset = PCRTrainDataset(
    train_queries, train_cands, tokenizer,
    chunk_size=CFG["chunk_size"],
    stride=CFG["chunk_stride"],
    max_chunks=CFG["max_chunks"]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    collate_fn=pcr_collate_fn,
    pin_memory=True,
    drop_last=(len(train_dataset) >= CFG["batch_size"] and PROFILE != "SMOKE")
)
print(f"✓ Train DataLoader initialized: {len(train_loader)} batches per epoch.")


In [ ]:
# 8. Non-Blocking Continuous Hub Checkpoint Sync & Crash Recovery
class ContinuousHubSyncManager:
    """
    Background thread manager that pushes checkpoints to Hugging Face Hub
    every save_steps without stalling GPU compute.
    """
    def __init__(self, repo_id: str, subfolder: str, token: str):
        self.repo_id = repo_id
        self.subfolder = subfolder
        self.token = token
        self.api = HfApi(token=token) if token else None

    def push_checkpoint_async(self, local_cp_dir: str, step: int):
        if not self.api:
            return
        def _upload():
            try:
                print(f"\n[HubSync] Starting background upload for checkpoint-{step}...")
                self.api.upload_folder(
                    folder_path=local_cp_dir,
                    path_in_repo=f"{self.subfolder}/checkpoint-{step}",
                    repo_id=self.repo_id,
                    commit_message=f"Auto-sync checkpoint-{step} [Step {step}]"
                )
                print(f"[HubSync] ✓ Checkpoint-{step} successfully synced to HF Hub!")
            except Exception as e:
                print(f"[HubSync] ⚠ Warning: Upload for checkpoint-{step} failed: {e}")

        thread = threading.Thread(target=_upload)
        thread.daemon = True
        thread.start()

    def push_final_artifacts(self, local_dir: str):
        if not self.api:
            return
        print(f"\n[HubSync] Pushing all final artifacts to {self.repo_id}/{self.subfolder}...")
        try:
            self.api.upload_folder(
                folder_path=local_dir,
                path_in_repo=self.subfolder,
                repo_id=self.repo_id,
                commit_message=f"Final Phase 7 artifacts for {EXPERIMENT_NAME}"
            )
            print("[HubSync] ✓ All artifacts successfully pushed to Hugging Face Hub!")
        except Exception as e:
            print(f"[HubSync] ⚠ Final upload error: {e}")

hub_manager = ContinuousHubSyncManager(HF_MASTER_REPO, HUB_SUBFOLDER, HF_TOKEN)


def find_and_restore_checkpoint(local_dir: str, hub_repo: str, hub_subfolder: str, token: str):
    """
    Checks local disk first. If empty or corrupted, queries HF Hub,
    downloads the latest checkpoint-{step}, and returns checkpoint state.
    Guaranteed to never fail with FileNotFoundError if an empty checkpoint folder exists.
    """
    import re
    # 1. Search for existing local checkpoints that actually contain a valid training_state.pt
    valid_local = []
    if os.path.exists(local_dir):
        for root, dirs, files in os.walk(local_dir):
            if "training_state.pt" in files:
                m = re.search(r"checkpoint-(\d+)", root)
                if m:
                    step_val = int(m.group(1))
                    state_path = os.path.join(root, "training_state.pt")
                    if os.path.isfile(state_path) and os.path.getsize(state_path) > 0:
                        valid_local.append((step_val, state_path))

    if valid_local:
        best_step, best_file = max(valid_local, key=lambda x: x[0])
        print(f"[Recovery] Found valid local checkpoint-{best_step}: {best_file}")
        try:
            return best_step, torch.load(best_file, map_location="cpu")
        except Exception as e:
            print(f"[Recovery] ⚠ Warning: Could not load local {best_file}: {e}")

    # 2. Remote HF Hub search
    if token:
        try:
            api = HfApi(token=token)
            files = api.list_repo_files(repo_id=hub_repo, repo_type="model", token=token)
            prefix = f"{hub_subfolder}/checkpoint-"
            hub_steps = []
            for f in files:
                if f.startswith(prefix) and "training_state.pt" in f:
                    rem = f[len(prefix):].split("/")[0]
                    if rem.isdigit():
                        hub_steps.append(int(rem))
            if hub_steps:
                latest_step = max(hub_steps)
                print(f"[Recovery] 🚀 CRASH RECOVERY ACTIVATED: Found remote checkpoint-{latest_step} on Hub!")
                target_local = os.path.join(local_dir, f"checkpoint-{latest_step}")
                os.makedirs(target_local, exist_ok=True)
                dl_dir = snapshot_download(
                    repo_id=hub_repo,
                    allow_patterns=[f"{hub_subfolder}/checkpoint-{latest_step}/*"],
                    local_dir=local_dir,
                    token=token
                )
                candidates = [
                    os.path.join(target_local, "training_state.pt"),
                    os.path.join(local_dir, hub_subfolder, f"checkpoint-{latest_step}", "training_state.pt"),
                    os.path.join(dl_dir, hub_subfolder, f"checkpoint-{latest_step}", "training_state.pt"),
                    os.path.join(dl_dir, f"checkpoint-{latest_step}", "training_state.pt")
                ]
                for cand in candidates:
                    if os.path.isfile(cand) and os.path.getsize(cand) > 0:
                        print(f"[Recovery] Restored training state from: {cand}")
                        return latest_step, torch.load(cand, map_location="cpu")
                for root, dirs, files in os.walk(local_dir):
                    if "training_state.pt" in files and f"checkpoint-{latest_step}" in root:
                        cand = os.path.join(root, "training_state.pt")
                        if os.path.getsize(cand) > 0:
                            print(f"[Recovery] Restored training state from: {cand}")
                            return latest_step, torch.load(cand, map_location="cpu")
        except Exception as e:
            print(f"[Recovery] Hub check notice: {e}")

    print("[Recovery] No valid previous checkpoint found. Starting fresh training from step 0.")
    return 0, None

In [ ]:
# 9. Pre-Flight Verification & Gradient Flow Assertions
print("Running mandatory 1-batch pre-flight gradient verification...")
assert len(train_loader) > 0, f"DataLoader has 0 batches! Dataset size: {len(train_dataset)}, batch_size: {CFG['batch_size']}"
model.train()

test_batch = next(iter(train_loader))
# Micro-slice to at most 2 samples to verify gradient flow safely without VRAM pressure
torch.cuda.empty_cache()
q_ids, q_attn, q_cmask, d_ids, d_attn, d_cmask = [t[:1].to(device) for t in test_batch]

with torch.amp.autocast("cuda", dtype=torch.float16):
    u, v = model(q_ids, q_attn, q_cmask, d_ids, d_attn, d_cmask)
    sim_matrix = torch.matmul(u, v.t()) / 0.05
    targets = torch.arange(u.size(0), device=device)
    loss = F.cross_entropy(sim_matrix, targets)

assert not torch.isnan(loss), "Pre-flight test loss produced NaN!"
loss.backward()

# Verify non-zero gradients on backbone layers
grad_norms = [p.grad.norm().item() for p in model.parameters() if p.grad is not None]
assert len(grad_norms) > 0, "No gradients were computed during backward pass!"
assert any(gn > 0.0 for gn in grad_norms), "Gradients are all zero!"

model.zero_grad()
del test_batch, q_ids, q_attn, q_cmask, d_ids, d_attn, d_cmask, u, v, sim_matrix, targets, loss
torch.cuda.empty_cache()
print("✓ Pre-flight verification passed! Gradients flow correctly through full architecture.")


In [ ]:
# 10. Supervised Multi-GPU Training Execution Loop
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# Wrap in DataParallel if dual GPUs are available
parallel_model = nn.DataParallel(model) if USE_MULTI_GPU else model

# Optimizer & Scheduler
PEAK_LR = 2.0e-5
MIN_LR = 2.0e-6
optimizer = optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=0.01, eps=1e-8)
scaler = torch.amp.GradScaler("cuda")

total_training_steps = len(train_loader) * CFG["train_epochs"]
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, total_training_steps), eta_min=MIN_LR)

# Check for crash resumption
start_step, recovered_state = find_and_restore_checkpoint(CHECKPOINT_DIR, HF_MASTER_REPO, HUB_SUBFOLDER, HF_TOKEN)
if recovered_state is not None:
    try:
        state_dict = recovered_state["model_state"]
        filtered_state = {
            k: v for k, v in state_dict.items()
            if not any(x in k for x in ["absmax", "quant_map", "nested_absmax", "nested_quant_map", "quant_state"])
        }
        missing_keys, unexpected_keys = model.load_state_dict(filtered_state, strict=False)
        print(f"✓ Model weights restored (strict=False). Missing: {len(missing_keys)}, Unexpected: {len(unexpected_keys)}")
        optimizer.load_state_dict(recovered_state["optimizer_state"])
        scheduler.load_state_dict(recovered_state["scheduler_state"])
        print(f"✓ Resumed training successfully from step {start_step}!")
    except Exception as e:
        print(f"⚠️ Warning: Could not resume checkpoint state cleanly ({e}). Starting fresh from step 0.")
        start_step = 0

# Training state tracking
global_step = start_step
history = {"steps": [], "train_loss": [], "dev_f1": []}
TAU = 0.05  # InfoNCE temperature
best_dev_f1 = -1.0

print(f"\nBeginning training across {CFG['train_epochs']} epoch(s) ({total_training_steps} total steps)...")
start_time = time.time()

for epoch in range(CFG["train_epochs"]):
    model.train()
    running_loss = 0.0

    for step, batch in enumerate(train_loader):
        global_step += 1
        q_ids, q_attn, q_cmask, d_ids, d_attn, d_cmask = [t.to(device) for t in batch]

        optimizer.zero_grad()

        # Dual-Tower Forward Pass under AMP
        with torch.amp.autocast("cuda", dtype=torch.float16):
            if USE_MULTI_GPU:
                u, v = parallel_model(q_ids, q_attn, q_cmask, d_ids, d_attn, d_cmask)
            else:
                u, v = model(q_ids, q_attn, q_cmask, d_ids, d_attn, d_cmask)

            # Symmetric InfoNCE Loss
            sim_q2d = torch.matmul(u, v.t()) / TAU
            sim_d2q = torch.matmul(v, u.t()) / TAU
            labels = torch.arange(u.size(0), device=device)
            loss = (F.cross_entropy(sim_q2d, labels) + F.cross_entropy(sim_d2q, labels)) / 2.0

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()

        # Step logging
        if global_step % 20 == 0 or global_step == 1:
            avg_loss = running_loss / max(1, (step + 1))
            current_lr = scheduler.get_last_lr()[0]
            print(f"Epoch [{epoch+1}/{CFG['train_epochs']}] | Step {global_step}/{total_training_steps} | "
                  f"Loss: {loss.item():.4f} (Avg: {avg_loss:.4f}) | LR: {current_lr:.2e}")

        # Periodic Evaluation & Continuous Cloud Sync
        if global_step % CFG["save_steps"] == 0 or global_step == total_training_steps:
            history["steps"].append(global_step)
            history["train_loss"].append(loss.item())

            # Save local checkpoint
            cp_folder = os.path.join(CHECKPOINT_DIR, f"checkpoint-{global_step}")
            os.makedirs(cp_folder, exist_ok=True)
            torch.save({
                "step": global_step,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "loss": loss.item()
            }, os.path.join(cp_folder, "training_state.pt"))

            # Keep only the latest local checkpoint to conserve Kaggle disk
            for cp in glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*")):
                if cp != cp_folder:
                    shutil.rmtree(cp, ignore_errors=True)

            # Continuous Non-blocking Push to HF Hub
            hub_manager.push_checkpoint_async(cp_folder, global_step)

print(f"\n✓ Training finished in {(time.time() - start_time) / 60:.2f} minutes.")


In [ ]:
# 11. Plot Training Loss Curve
if history["steps"]:
    plt.figure(figsize=(9, 4.5))
    plt.plot(history["steps"], history["train_loss"], label="InfoNCE Training Loss", color="royalblue", lw=2)
    plt.xlabel("Global Step")
    plt.ylabel("Loss")
    plt.title(f"Phase 7 Sub-01 Training Loss - {EXPERIMENT_NAME}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(LOSS_PLOT_PATH, dpi=150)
    plt.show()
    print(f"Loss plot saved to: {LOSS_PLOT_PATH}")


In [ ]:
# 12. Dual-GPU Accelerated Candidate Pre-Encoding & Top-7 Test Inference
print("=== High-Speed Candidate Pre-Encoding & Test Inference ===")
model.eval()

# Helper to encode a large dictionary of legal documents into a cached tensor
@torch.no_grad()
def precompute_document_embeddings(doc_dict: dict, batch_size: int = 16) -> Tuple[List[str], torch.Tensor]:
    doc_ids = list(doc_dict.keys())
    embeddings = []
    
    for i in range(0, len(doc_ids), batch_size):
        b_ids = doc_ids[i : i + batch_size]
        b_tensors = [
            process_text_into_chunks(
                doc_dict[did]["text"], tokenizer,
                chunk_size=CFG["chunk_size"],
                stride=CFG["chunk_stride"],
                max_chunks=CFG["max_chunks"]
            )
            for did in b_ids
        ]
        
        input_ids = torch.stack([t[0] for t in b_tensors]).to(device)
        attention_mask = torch.stack([t[1] for t in b_tensors]).to(device)
        chunk_mask = torch.stack([t[2] for t in b_tensors]).to(device)
        
        with torch.amp.autocast("cuda", dtype=torch.float16):
            if USE_MULTI_GPU:
                vecs = parallel_model(input_ids, attention_mask, chunk_mask)
            else:
                vecs = model(input_ids, attention_mask, chunk_mask)
            
        embeddings.append(vecs.cpu())
        
        if (i + batch_size) % 128 == 0 or (i + len(b_ids)) == len(doc_ids):
            print(f"  Processed {min(i + batch_size, len(doc_ids))}/{len(doc_ids)} documents...")
            
    return doc_ids, torch.cat(embeddings, dim=0)

# 1. Pre-encode all test candidate documents once
print(f"Pre-encoding {len(test_cands):,} candidate documents...")
cand_ids, V_cands = precompute_document_embeddings(test_cands, batch_size=16)
V_cands = V_cands.to(device)
print(f"✓ Candidate Matrix cached: {V_cands.shape} (dtype: {V_cands.dtype})")

# 2. Encode test query cases
print(f"Pre-encoding {len(test_queries):,} test query cases...")
q_ids, U_queries = precompute_document_embeddings(test_queries, batch_size=16)
U_queries = U_queries.to(device)
print(f"✓ Query Matrix cached: {U_queries.shape}")

# 3. Compute Top 7 Predictions via Matrix Multiplication
print("Computing similarity matrix and ranking Top 7 candidates...")
# Cosine similarity matrix: S = U * V^T (vectors are already unit normalized)
sim_matrix = torch.matmul(U_queries, V_cands.t())  # [Num_Queries, Num_Candidates]
k_retrieval = min(7, V_cands.size(0))
top_indices = torch.topk(sim_matrix, k=k_retrieval, dim=-1).indices.cpu().tolist()

predictions = {}
for qid, indices in zip(q_ids, top_indices):
    preds = [cand_ids[idx] for idx in indices]
    while len(preds) < 7 and cand_ids:
        for cid in cand_ids:
            if cid not in preds:
                preds.append(cid)
            if len(preds) == 7:
                break
    predictions[str(qid)] = preds

print(f"✓ Generated predictions for {len(predictions):,} queries.")


In [ ]:
# 13. Strict CodaBench Packaging & Hugging Face Hub Final Synchronization
print("=== Strict CodaBench Verification ===")

# Verification Assertions
assert len(predictions) == len(test_queries), f"Query count mismatch: {len(predictions)} vs {len(test_queries)}"
all_cand_set = set(test_cands.keys())

for qid, cands in predictions.items():
    assert len(cands) == 7, f"Query {qid} does not have exactly 7 predictions!"
    assert len(set(cands)) == 7, f"Query {qid} contains duplicate predictions: {cands}"
    for cid in cands:
        assert cid in all_cand_set, f"Predicted candidate {cid} not in official candidate pool!"

print("✓ All strict format assertions passed successfully!")

# Write prediction.json
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2)
print(f"✓ Saved {PREDICTION_PATH} ({os.path.getsize(PREDICTION_PATH) / 1024:.1f} KB)")

# Package submission.zip directly at root
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")
print(f"✓ Packaged {SUBMISSION_ZIP}")

# Verify ZIP structure
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zipf:
    names = zipf.namelist()
    assert "prediction.json" in names, "ERROR: prediction.json not at root of ZIP!"
    assert len(names) == 1, f"ERROR: ZIP contains unexpected extra files: {names}"
    print(f"✓ Verification confirmed: ZIP contents: {names}")

# Save final model weights
final_weights_path = os.path.join(OUTPUT_DIR, "model_weights.pt")
torch.save(model.state_dict(), final_weights_path)
print(f"✓ Saved final model weights to: {final_weights_path}")

# Push all final deliverables to Hugging Face Hub
hub_manager.push_final_artifacts(OUTPUT_DIR)

print("\n" + "="*70)
print(f"PHASE 7 SUBMISSION 01 COMPLETED SUCCESSFULLY!")
print(f"  Submission File: {SUBMISSION_ZIP}")
print(f"  Predictions:     {PREDICTION_PATH}")
print(f"  Master Hub Repo: https://huggingface.co/{HF_MASTER_REPO}/tree/main/{HUB_SUBFOLDER}")
print("="*70)
